# 07_03G — Modelo híbrido com todas as famílias de features — Early Stage

**Objetivo:** testar se a combinação de features tabulares, históricas, encoders, texto e embeddings de grafo melhora a priorização de processos com risco de perda para o banco.

Este notebook consolida as principais famílias testadas anteriormente:

1. **Features tabulares base**: numéricas e categóricas originais já liberadas para modelagem.
2. **Features históricas / target encoding temporal**: variáveis `fe_*hist*`, `*_loss_rate*`, `*_gain_rate*`, `*_smooth*`, etc.
3. **Encoders supervisionados seguros**: `MEstimateEncoder`, `CatBoostEncoder`, `JamesSteinEncoder`, quando `category_encoders` estiver disponível.
4. **Texto**: TF-IDF sobre colunas textuais disponíveis, sem usar petição inicial.
5. **Grafo**: embeddings Node2Vec com `pecanpy`, quando disponível, ou fallback com `TruncatedSVD`.
6. **Modelos**: Random Forest, HistGradientBoosting, LightGBM e XGBoost, respeitando disponibilidade local.

> Importante: o notebook não usa `catboost` como modelo. O `CatBoostEncoder` do `category_encoders` é apenas um encoder e pode ser desligado caso compliance também restrinja esse módulo.

## 1. Imports, configurações e utilitários

In [ ]:
import os
import re
import gc
import json
import time
import math
import warnings
from pathlib import Path
from itertools import combinations
from collections import defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.validation import check_is_fitted

try:
    from IPython.display import display
except Exception:
    display = print

# Modelos opcionais
try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

# Encoders opcionais
try:
    import category_encoders as ce
    HAS_CATEGORY_ENCODERS = True
except Exception:
    HAS_CATEGORY_ENCODERS = False

# PecanPy opcional para Node2Vec rápido
try:
    from pecanpy import pecanpy
    HAS_PECANPY = True
except Exception:
    HAS_PECANPY = False

# Gensim opcional para treinar Word2Vec em walks do PecanPy
try:
    from gensim.models import Word2Vec
    HAS_GENSIM = True
except Exception:
    HAS_GENSIM = False

print("HAS_LIGHTGBM:", HAS_LIGHTGBM)
print("HAS_XGBOOST:", HAS_XGBOOST)
print("HAS_CATEGORY_ENCODERS:", HAS_CATEGORY_ENCODERS)
print("HAS_PECANPY:", HAS_PECANPY)
print("HAS_GENSIM:", HAS_GENSIM)

In [ ]:
# ==============================
# Configurações principais
# ==============================

RUN_MODE = "fast"  # "fast" ou "full"
RANDOM_STATE = 42
TARGET_COL = None  # deixar None para detecção automática
DATE_COL = None    # deixar None para detecção automática
ID_COL = None      # deixar None para detecção automática
VALUE_COL = None   # deixar None para detecção automática

# Texto
USE_TEXT_FEATURES = True
TFIDF_MAX_FEATURES = 1500 if RUN_MODE == "fast" else 3000
TFIDF_MIN_DF = 20 if RUN_MODE == "fast" else 10

# Grafo / Node2Vec
USE_GRAPH_FEATURES = True
GRAPH_METHOD = "pecanpy"  # "pecanpy", "svd", ou "auto"
N2V_DIM = 32 if RUN_MODE == "fast" else 64
N2V_WALK_LENGTH = 16 if RUN_MODE == "fast" else 32
N2V_NUM_WALKS = 4 if RUN_MODE == "fast" else 8
N2V_WINDOW = 5
N2V_EPOCHS = 5 if RUN_MODE == "fast" else 10
N2V_P = 1.0
N2V_Q = 1.0
MIN_ENTITY_FREQ = 20 if RUN_MODE == "fast" else 5
MAX_ENTITY_VALUES_PER_COL = 5000 if RUN_MODE == "fast" else 20000

# Modelagem
TOP_KS = (0.01, 0.05, 0.10, 0.20)
N_SPLITS_WF = 3 if RUN_MODE == "fast" else 5

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"
MODELS_DIR = BASE_DIR / "outputs" / "models"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

In [ ]:
def save_report(df: pd.DataFrame, filename: str) -> Path:
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Relatório salvo em: {path}")
    return path


def load_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f"Lendo: {p}")
            if p.suffix.lower() == ".parquet":
                return pd.read_parquet(p)
            if p.suffix.lower() in [".csv", ".txt"]:
                return pd.read_csv(p)
            raise ValueError(f"Extensão não suportada: {p}")
    raise FileNotFoundError("Nenhum arquivo encontrado entre:\n" + "\n".join(map(str, paths)))


def normalize_colname(c: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(c).strip().lower())


def infer_target_col(df: pd.DataFrame) -> str:
    candidates = [
        "target_banco_perdeu", "banco_perdeu", "target_perda", "perda", "target",
        "y", "label", "desfecho_binario", "target_corrigido", "target_semantica"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    # tenta achar coluna binária com nome sugestivo
    for c in df.columns:
        lc = c.lower()
        if any(s in lc for s in ["perdeu", "perda", "target", "desfecho"]):
            vals = set(pd.Series(df[c]).dropna().unique())
            if vals.issubset({0, 1, False, True}):
                return c
    raise ValueError("Não consegui inferir TARGET_COL. Defina TARGET_COL manualmente.")


def infer_date_col(df: pd.DataFrame) -> str:
    candidates = [
        "Data_Distribuicao", "data_distribuicao", "data_inicio_processo", "DataInicio", "data_inicio",
        "dt_inicio", "date_ref", "data_ref", "DataCadastro", "data_cadastro"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if "data" in lc or lc.startswith("dt_") or "date" in lc:
            try:
                pd.to_datetime(df[c], errors="raise")
                return c
            except Exception:
                pass
    raise ValueError("Não consegui inferir DATE_COL. Defina DATE_COL manualmente.")


def infer_id_col(df: pd.DataFrame) -> str:
    candidates = [
        "Numerodistribuicao", "numero_processo", "processo_id", "id_processo", "Identificador", "identificador",
        "ID", "id"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return df.index.name if df.index.name else "__row_id__"


def get_value_col(df: pd.DataFrame) -> str | None:
    candidates = [
        "fe_valor_ajuizado", "valor_ajuizado", "Valorajuizado", "Valor_Ajuizado",
        "valor_causa", "Valor_Causa", "valor_ajustado", "valor_exposicao"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if "valor" in lc and any(s in lc for s in ["ajuiz", "causa", "expos", "perda"]):
            if pd.api.types.is_numeric_dtype(df[c]):
                return c
    return None


def safe_auc(y_true, proba):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, proba)


def safe_ap(y_true, proba):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, proba)

## 2. Carregar Dev/Holdout gerados pelo 07_01

O notebook tenta localizar automaticamente os arquivos mais prováveis. Caso seus nomes sejam diferentes, ajuste a lista `DEV_CANDIDATES` e `HOLDOUT_CANDIDATES`.

In [ ]:
DEV_CANDIDATES = [
    PROCESSED_DIR / "abt_early_stage_v1_dev.parquet",
    PROCESSED_DIR / "abt_early_stage_dev.parquet",
    PROCESSED_DIR / "abt_early_stage_v1_corrigido_dev.parquet",
    PROCESSED_DIR / "abt_early_stage_corrigido_dev.parquet",
]

HOLDOUT_CANDIDATES = [
    PROCESSED_DIR / "abt_early_stage_v1_holdout.parquet",
    PROCESSED_DIR / "abt_early_stage_holdout.parquet",
    PROCESSED_DIR / "abt_early_stage_v1_corrigido_holdout.parquet",
    PROCESSED_DIR / "abt_early_stage_corrigido_holdout.parquet",
]

df_dev = load_first_existing(DEV_CANDIDATES)
df_holdout = load_first_existing(HOLDOUT_CANDIDATES)

# ID auxiliar caso não exista coluna de ID
if "__row_id__" not in df_dev.columns:
    df_dev["__row_id__"] = [f"dev_{i}" for i in range(len(df_dev))]
if "__row_id__" not in df_holdout.columns:
    df_holdout["__row_id__"] = [f"holdout_{i}" for i in range(len(df_holdout))]

TARGET_COL = TARGET_COL or infer_target_col(df_dev)
DATE_COL = DATE_COL or infer_date_col(df_dev)
ID_COL = ID_COL or infer_id_col(df_dev)
VALUE_COL = VALUE_COL or get_value_col(df_dev)

for df in [df_dev, df_holdout]:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df[TARGET_COL] = df[TARGET_COL].astype(int)

summary_load = pd.DataFrame([{
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "target_col": TARGET_COL,
    "date_col": DATE_COL,
    "id_col": ID_COL,
    "value_col": VALUE_COL,
    "taxa_perda_dev": df_dev[TARGET_COL].mean(),
    "taxa_perda_holdout": df_holdout[TARGET_COL].mean(),
    "min_date_dev": df_dev[DATE_COL].min(),
    "max_date_dev": df_dev[DATE_COL].max(),
    "min_date_holdout": df_holdout[DATE_COL].min(),
    "max_date_holdout": df_holdout[DATE_COL].max(),
}])
save_report(summary_load, "90_3g_load_summary.csv")
display(summary_load)

## 3. Remoção defensiva de leakage e definição de famílias de features

A lista abaixo remove campos de encerramento, sentença, condenação, acordo, resultado final e datas posteriores ao desfecho. Ajuste conforme seu dicionário de dados.

In [ ]:
LEAKAGE_PATTERNS = [
    "dataencerr", "data_encerr", "encerramento", "motivo_encerr", "resultado_final",
    "sentenca", "sentença", "condenacao", "condenação", "valor_do_acordo",
    "acordo_pos", "acordo pós", "status_final", "desfecho_text", "desfecho_final",
    "baixado", "arquivado", "transitado", "cumprimento_sentenca", "cumprimento_sentença",
]

ALWAYS_EXCLUDE = {TARGET_COL, DATE_COL}
if ID_COL:
    # ID será preservado para ranking, mas não entra como feature numérica/categórica pura.
    ALWAYS_EXCLUDE.add(ID_COL)


def is_leakage_col(c: str) -> bool:
    lc = c.lower()
    return any(p in lc for p in LEAKAGE_PATTERNS)

candidate_cols = [c for c in df_dev.columns if c not in ALWAYS_EXCLUDE]
leakage_cols = [c for c in candidate_cols if is_leakage_col(c)]
usable_cols = [c for c in candidate_cols if c not in leakage_cols]

print("Qtd colunas totais:", len(df_dev.columns))
print("Qtd leakage removidas:", len(leakage_cols))
print("Qtd candidatas pós-leakage:", len(usable_cols))
print("Exemplos leakage removidas:", leakage_cols[:30])

In [ ]:
def looks_text_col(s: pd.Series, col: str) -> bool:
    lc = col.lower()
    if any(k in lc for k in ["texto", "descricao", "descrição", "historico", "histórico", "observacao", "observação", "ementa", "assunto_texto"]):
        return True
    if s.dtype == "object":
        sample = s.dropna().astype(str).head(500)
        if len(sample) == 0:
            return False
        avg_len = sample.str.len().mean()
        return avg_len >= 40
    return False


def is_historical_feature(c: str) -> bool:
    lc = c.lower()
    tokens = [
        "hist_", "_hist", "historical", "loss_rate", "gain_rate", "taxa_perda", "taxa_ganho",
        "smooth", "expanding", "rolling", "mean_across_groups", "max_across_groups", "min_across_groups",
        "target_enc", "target_encoding", "woe", "mestimate", "james", "catboost_encoder"
    ]
    return any(t in lc for t in tokens)


def is_numeric_col(df, c):
    return pd.api.types.is_numeric_dtype(df[c]) and c not in ALWAYS_EXCLUDE

text_cols = [c for c in usable_cols if looks_text_col(df_dev[c], c)]
historical_cols = [c for c in usable_cols if is_historical_feature(c) and is_numeric_col(df_dev, c)]
numeric_base_cols = [c for c in usable_cols if is_numeric_col(df_dev, c) and c not in historical_cols]

categorical_cols = []
for c in usable_cols:
    if c in text_cols or c in historical_cols or c in numeric_base_cols:
        continue
    if df_dev[c].dtype == "object" or str(df_dev[c].dtype).startswith("category") or str(df_dev[c].dtype).startswith("bool"):
        categorical_cols.append(c)

# Colunas candidatas para grafo: categóricas com alguma cardinalidade e relevância de entidade/contexto
GRAPH_NAME_HINTS = [
    "uf", "comarca", "vara", "orgao", "órgão", "juiz", "juiz", "escritorio", "escritório",
    "advogado", "assunto", "acao", "ação", "produto", "carteira", "fase", "classe", "tipo",
    "status", "segmento", "regional"
]

graph_entity_cols = []
for c in categorical_cols:
    lc = c.lower()
    nunique = df_dev[c].nunique(dropna=True)
    if nunique >= 2 and any(h in lc for h in GRAPH_NAME_HINTS):
        graph_entity_cols.append(c)

# Limites para evitar estouro local
if RUN_MODE == "fast":
    graph_entity_cols = graph_entity_cols[:16]

feature_inventory = pd.DataFrame({
    "feature_family": ["numeric_base", "historical_target_like", "categorical", "text", "graph_entity_cols"],
    "qtd_cols": [len(numeric_base_cols), len(historical_cols), len(categorical_cols), len(text_cols), len(graph_entity_cols)],
    "examples": [
        numeric_base_cols[:15], historical_cols[:15], categorical_cols[:15], text_cols[:10], graph_entity_cols[:20]
    ]
})
save_report(feature_inventory, "91_3g_feature_inventory.csv")
display(feature_inventory)

## 4. Funções de métricas, walk-forward e top-k

In [ ]:
def best_f1_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    # precision/recall têm 1 elemento a mais que thresholds
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    if len(thresholds) == 0:
        return 0.5, precision[0], recall[0], f1[0]
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx]), float(precision[idx]), float(recall[idx]), float(f1[idx])


def threshold_metrics_at(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": pred.mean(),
    }


def eval_binary(y_true, proba, prefix=""):
    roc = safe_auc(y_true, proba)
    pr = safe_ap(y_true, proba)
    best_thr, best_p, best_r, best_f1 = best_f1_threshold(y_true, proba)
    m05 = threshold_metrics_at(y_true, proba, 0.5)
    out = {
        f"{prefix}roc_auc_perda": roc,
        f"{prefix}pr_auc_perda": pr,
        f"{prefix}taxa_perda": float(np.mean(y_true)),
        f"{prefix}taxa_ganho": float(1 - np.mean(y_true)),
        f"{prefix}best_f1_threshold_perda": best_thr,
        f"{prefix}best_f1_precision_perda": best_p,
        f"{prefix}best_f1_recall_perda": best_r,
        f"{prefix}best_f1_perda": best_f1,
        f"{prefix}precision_perda_t05": m05["precision_perda"],
        f"{prefix}recall_perda_t05": m05["recall_perda"],
        f"{prefix}f1_perda_t05": m05["f1_perda"],
        f"{prefix}pred_perda_rate_t05": m05["pred_perda_rate"],
    }
    return out


def make_walk_forward_folds(df, date_col, n_splits=3, min_train_frac=0.45):
    d = df.sort_values(date_col).reset_index(drop=True)
    unique_dates = np.array(sorted(d[date_col].dropna().unique()))
    if len(unique_dates) < n_splits + 2:
        raise ValueError("Poucas datas únicas para walk-forward.")
    cut_positions = np.linspace(int(len(unique_dates) * min_train_frac), len(unique_dates) - 1, n_splits + 1, dtype=int)
    folds = []
    for i in range(n_splits):
        train_end = unique_dates[cut_positions[i]]
        val_end = unique_dates[cut_positions[i + 1]]
        train_idx = d.index[d[date_col] <= train_end].to_numpy()
        val_idx = d.index[(d[date_col] > train_end) & (d[date_col] <= val_end)].to_numpy()
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue
        folds.append({
            "fold": i + 1,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "train_start_date": d.loc[train_idx, date_col].min(),
            "train_end_date": d.loc[train_idx, date_col].max(),
            "val_start_date": d.loc[val_idx, date_col].min(),
            "val_end_date": d.loc[val_idx, date_col].max(),
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
        })
    return d, folds


def topk_risco_perda_metrics(y_true, proba_perda, ks=TOP_KS):
    df = pd.DataFrame({"y": np.asarray(y_true), "proba": np.asarray(proba_perda)})
    df = df.sort_values("proba", ascending=False).reset_index(drop=True)
    base = df["y"].mean()
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(df) * k)))
        top = df.head(n_top)
        precision_at_k = top["y"].mean()
        recall_at_k = top["y"].sum() / max(df["y"].sum(), 1)
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_at_k,
            "recall_perda_at_k": recall_at_k,
            "lift_perda_at_k": precision_at_k / base if base > 0 else np.nan,
            "taxa_perda_base": base,
        })
    return pd.DataFrame(rows)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=TOP_KS):
    df = pd.DataFrame({
        "y": np.asarray(y_true),
        "proba": np.asarray(proba_perda),
        "valor": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0).clip(lower=0).to_numpy(),
    })
    df["valor_esperado_perda"] = df["proba"] * df["valor"]
    df["valor_perda_real"] = df["y"] * df["valor"]
    df = df.sort_values("valor_esperado_perda", ascending=False).reset_index(drop=True)
    base = df["y"].mean()
    total_valor = df["valor"].sum()
    total_perdas = df["valor_perda_real"].sum()
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(df) * k)))
        top = df.head(n_top)
        precision_at_k = top["y"].mean()
        recall_at_k = top["y"].sum() / max(df["y"].sum(), 1)
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_at_k,
            "recall_perda_at_k": recall_at_k,
            "lift_perda_at_k": precision_at_k / base if base > 0 else np.nan,
            "taxa_perda_base": base,
            "valor_ajuizado_top": top["valor"].sum(),
            "share_valor_ajuizado_total": top["valor"].sum() / total_valor if total_valor > 0 else np.nan,
            "valor_ajuizado_perdas_top": top["valor_perda_real"].sum(),
            "share_valor_perdas_total": top["valor_perda_real"].sum() / total_perdas if total_perdas > 0 else np.nan,
        })
    return pd.DataFrame(rows)

## 5. Graph features — Node2Vec com PecanPy ou fallback SVD

Estratégia adotada para evitar leakage temporal:

- o grafo é treinado apenas no bloco de treino de cada fold;
- os embeddings são aprendidos para os nós-entidade;
- cada processo recebe a média dos embeddings das entidades conectadas a ele;
- processos futuros podem ser representados desde que compartilhem entidades conhecidas no treino;
- entidades novas no período futuro recebem vetor zero.

Isso evita treinar embeddings usando a estrutura do holdout, mas preserva a capacidade de pontuar casos novos.

In [ ]:
def clean_entity_value(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == "" or s.lower() in ["nan", "none", "null"]:
        return None
    s = re.sub(r"\s+", " ", s)
    return s[:160]


def build_entity_tokens(df: pd.DataFrame, entity_cols, min_freq=MIN_ENTITY_FREQ, max_values_per_col=MAX_ENTITY_VALUES_PER_COL):
    # Calcula valores frequentes por coluna apenas no treino
    allowed = {}
    for c in entity_cols:
        vc = df[c].map(clean_entity_value).value_counts(dropna=True)
        vc = vc[vc >= min_freq].head(max_values_per_col)
        allowed[c] = set(vc.index)

    row_tokens = []
    for _, row in df[entity_cols].iterrows():
        tokens = []
        for c in entity_cols:
            v = clean_entity_value(row[c])
            if v is not None and v in allowed[c]:
                tokens.append(f"{c}={v}")
        row_tokens.append(tokens)
    return row_tokens, allowed


def transform_entity_tokens(df: pd.DataFrame, entity_cols, allowed):
    row_tokens = []
    for _, row in df[entity_cols].iterrows():
        tokens = []
        for c in entity_cols:
            v = clean_entity_value(row[c])
            if v is not None and c in allowed and v in allowed[c]:
                tokens.append(f"{c}={v}")
        row_tokens.append(tokens)
    return row_tokens


def build_weighted_entity_edges(row_tokens):
    weights = defaultdict(float)
    for toks in row_tokens:
        toks = sorted(set(toks))
        for a, b in combinations(toks, 2):
            if a != b:
                weights[(a, b)] += 1.0
    return weights


def write_edgelist(weights, path: Path):
    nodes = set()
    with open(path, "w", encoding="utf-8") as f:
        for (a, b), w in weights.items():
            nodes.add(a); nodes.add(b)
            f.write(f"{a}\t{b}\t{float(w)}\n")
    return nodes


def fit_entity_embeddings_pecanpy(row_tokens, dim=N2V_DIM, walk_length=N2V_WALK_LENGTH, num_walks=N2V_NUM_WALKS,
                                  window=N2V_WINDOW, epochs=N2V_EPOCHS, p=N2V_P, q=N2V_Q, random_state=RANDOM_STATE):
    if not (HAS_PECANPY and HAS_GENSIM):
        raise ImportError("PecanPy ou Gensim não disponíveis.")

    tmp_dir = REPORTS_DIR / "_tmp_graph"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    edge_path = tmp_dir / f"pecanpy_edges_{int(time.time()*1000)}.edg"

    weights = build_weighted_entity_edges(row_tokens)
    nodes = write_edgelist(weights, edge_path)
    if len(weights) == 0 or len(nodes) < 2:
        raise ValueError("Grafo vazio ou insuficiente para PecanPy.")

    g = pecanpy.SparseOTF(p=p, q=q, workers=max(1, os.cpu_count() or 1), verbose=False)
    g.read_edg(str(edge_path), weighted=True, directed=False)
    walks = g.simulate_walks(num_walks=num_walks, walk_length=walk_length)
    walks = [[str(x) for x in walk] for walk in walks]

    w2v = Word2Vec(
        sentences=walks,
        vector_size=dim,
        window=window,
        min_count=1,
        sg=1,
        workers=max(1, os.cpu_count() or 1),
        epochs=epochs,
        seed=random_state,
    )
    emb = {node: w2v.wv[node].astype(np.float32) for node in w2v.wv.index_to_key}
    meta = {"method": "pecanpy", "qtd_edges": len(weights), "qtd_nodes": len(emb), "dim": dim}
    try:
        edge_path.unlink(missing_ok=True)
    except Exception:
        pass
    return emb, meta


def fit_entity_embeddings_svd(train_tokens, dim=N2V_DIM):
    # Cria matriz processo x entidade e reduz por SVD; embedding de entidade = componentes transpostos.
    vocab = sorted({t for toks in train_tokens for t in toks})
    idx = {t: i for i, t in enumerate(vocab)}
    if len(vocab) < 2:
        emb = {t: np.zeros(dim, dtype=np.float32) for t in vocab}
        return emb, {"method": "svd_fallback_empty", "qtd_edges": 0, "qtd_nodes": len(vocab), "dim": dim}

    from scipy import sparse
    rows, cols, data = [], [], []
    for r, toks in enumerate(train_tokens):
        for t in set(toks):
            if t in idx:
                rows.append(r); cols.append(idx[t]); data.append(1.0)
    X = sparse.csr_matrix((data, (rows, cols)), shape=(len(train_tokens), len(vocab)))
    n_comp = min(dim, max(1, min(X.shape) - 1))
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    svd.fit(X)
    ent = svd.components_.T.astype(np.float32)
    if n_comp < dim:
        ent = np.pad(ent, ((0, 0), (0, dim - n_comp)), mode="constant")
    emb = {t: ent[i] for t, i in idx.items()}
    return emb, {"method": "svd_fallback", "qtd_edges": int(X.nnz), "qtd_nodes": len(vocab), "dim": dim}


def row_tokens_to_matrix(row_tokens, entity_emb, dim=N2V_DIM, prefix="n2v"):
    arr = np.zeros((len(row_tokens), dim), dtype=np.float32)
    for i, toks in enumerate(row_tokens):
        vecs = [entity_emb[t] for t in toks if t in entity_emb]
        if vecs:
            arr[i, :] = np.mean(vecs, axis=0)
    cols = [f"{prefix}_{j:03d}" for j in range(dim)]
    return pd.DataFrame(arr, columns=cols)


def make_graph_features(train_df, apply_df, entity_cols, graph_method="auto"):
    if not USE_GRAPH_FEATURES or len(entity_cols) == 0:
        empty_train = pd.DataFrame(index=train_df.index)
        empty_apply = pd.DataFrame(index=apply_df.index)
        return empty_train, empty_apply, {"method": "disabled", "qtd_edges": 0, "qtd_nodes": 0, "dim": 0}

    train_tokens, allowed = build_entity_tokens(train_df, entity_cols)
    apply_tokens = transform_entity_tokens(apply_df, entity_cols, allowed)

    method = graph_method
    if method == "auto":
        method = "pecanpy" if (HAS_PECANPY and HAS_GENSIM) else "svd"

    try:
        if method == "pecanpy":
            entity_emb, meta = fit_entity_embeddings_pecanpy(train_tokens)
        else:
            entity_emb, meta = fit_entity_embeddings_svd(train_tokens)
    except Exception as e:
        print(f"Fallback para SVD. Motivo: {repr(e)}")
        entity_emb, meta = fit_entity_embeddings_svd(train_tokens)

    Xg_train = row_tokens_to_matrix(train_tokens, entity_emb, dim=N2V_DIM, prefix="n2v")
    Xg_apply = row_tokens_to_matrix(apply_tokens, entity_emb, dim=N2V_DIM, prefix="n2v")
    Xg_train.index = train_df.index
    Xg_apply.index = apply_df.index
    return Xg_train, Xg_apply, meta

## 6. Preparação de datasets híbridos por fold

A função abaixo monta a matriz final combinando:

- colunas numéricas base;
- features históricas/target encoding temporal já existentes;
- categóricas para encoders;
- texto via TF-IDF;
- embeddings de grafo.

In [ ]:
class TextConcatTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, text_cols):
        self.text_cols = text_cols
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if len(self.text_cols) == 0:
            return np.array([""] * len(X))
        X_ = pd.DataFrame(X, columns=self.text_cols) if not isinstance(X, pd.DataFrame) else X
        return X_[self.text_cols].fillna("").astype(str).agg(" ".join, axis=1).values


def build_preprocessor(preprocess_mode, num_cols, cat_cols, text_cols, use_text=True):
    transformers = []

    if num_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            num_cols,
        ))

    if cat_cols:
        if preprocess_mode == "onehot":
            cat_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20 if RUN_MODE == "fast" else 5)),
            ])
        else:
            cat_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ])
        transformers.append(("cat", cat_pipe, cat_cols))

    if use_text and text_cols:
        text_pipe = Pipeline([
            ("concat", TextConcatTransformer(text_cols)),
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                min_df=TFIDF_MIN_DF,
                ngram_range=(1, 2),
                strip_accents="unicode",
                lowercase=True,
            )),
        ])
        transformers.append(("text", text_pipe, text_cols))

    if not transformers:
        raise ValueError("Nenhuma feature disponível para o preprocessor.")

    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)


def get_model_candidates():
    models = []

    # Baseline linear útil para texto/esparsidade
    models.append((
        "logistic_onehot_hybrid",
        "onehot",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="saga",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ))

    models.append((
        "random_forest_onehot_hybrid",
        "onehot",
        RandomForestClassifier(
            n_estimators=250 if RUN_MODE == "fast" else 500,
            max_depth=None,
            min_samples_leaf=20 if RUN_MODE == "fast" else 10,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ))

    models.append((
        "hgb_ordinal_hybrid",
        "ordinal",
        HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            max_iter=250 if RUN_MODE == "fast" else 500,
            random_state=RANDOM_STATE,
        )
    ))

    if HAS_LIGHTGBM:
        models.append((
            "lgbm_ordinal_hybrid",
            "ordinal",
            LGBMClassifier(
                n_estimators=400 if RUN_MODE == "fast" else 900,
                learning_rate=0.04,
                num_leaves=31,
                min_child_samples=50,
                subsample=0.9,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=1.0,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )
        ))

    if HAS_XGBOOST:
        models.append((
            "xgb_ordinal_hybrid",
            "ordinal",
            XGBClassifier(
                n_estimators=350 if RUN_MODE == "fast" else 800,
                learning_rate=0.04,
                max_depth=4,
                min_child_weight=10,
                subsample=0.9,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=2.0,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        ))

    return models


def predict_proba_positive(pipe, X):
    proba = pipe.predict_proba(X)
    if proba.ndim == 2:
        return proba[:, 1]
    return proba

## 7. Encoders supervisionados avançados opcionais

Esta seção adiciona colunas codificadas com `category_encoders`, usando apenas o treino de cada fold para evitar leakage. Esses encoders entram como features numéricas adicionais.

In [ ]:
def fit_transform_category_encoders(X_train, y_train, X_apply, cat_cols, max_cols=30):
    if not HAS_CATEGORY_ENCODERS or not cat_cols:
        return pd.DataFrame(index=X_train.index), pd.DataFrame(index=X_apply.index), []

    # Para custo local, usa colunas categóricas com maior cardinalidade informativa, limitadas.
    chosen = []
    for c in cat_cols:
        nun = X_train[c].nunique(dropna=True)
        if 2 <= nun <= 2000:
            chosen.append(c)
    chosen = chosen[:max_cols]
    if not chosen:
        return pd.DataFrame(index=X_train.index), pd.DataFrame(index=X_apply.index), []

    encoders = []
    # MEstimate costuma ser estável para target encoding regularizado
    encoders.append(("mestimate", ce.MEstimateEncoder(cols=chosen, m=20.0, handle_unknown="value", handle_missing="value")))
    # CatBoostEncoder é encoder, não modelo CatBoost; desligue se compliance restringir o nome/módulo.
    encoders.append(("catboost", ce.CatBoostEncoder(cols=chosen, random_state=RANDOM_STATE, handle_unknown="value", handle_missing="value")))
    encoders.append(("james_stein", ce.JamesSteinEncoder(cols=chosen, handle_unknown="value", handle_missing="value")))

    train_parts = []
    apply_parts = []
    used = []
    for name, enc in encoders:
        try:
            Xt = enc.fit_transform(X_train[chosen], y_train)
            Xa = enc.transform(X_apply[chosen])
            Xt.columns = [f"ce_{name}_{c}" for c in Xt.columns]
            Xa.columns = Xt.columns
            train_parts.append(Xt.reset_index(drop=True))
            apply_parts.append(Xa.reset_index(drop=True))
            used.append(name)
        except Exception as e:
            print(f"Encoder {name} falhou: {repr(e)}")

    if not train_parts:
        return pd.DataFrame(index=X_train.index), pd.DataFrame(index=X_apply.index), []

    Xce_train = pd.concat(train_parts, axis=1)
    Xce_apply = pd.concat(apply_parts, axis=1)
    Xce_train.index = X_train.index
    Xce_apply.index = X_apply.index
    return Xce_train, Xce_apply, used

## 8. Walk-forward no Dev — modelo híbrido completo

In [ ]:
df_dev_sorted, folds = make_walk_forward_folds(df_dev, DATE_COL, n_splits=N_SPLITS_WF)
fold_summary_rows = []

for f in folds:
    tr = df_dev_sorted.loc[f["train_idx"]]
    va = df_dev_sorted.loc[f["val_idx"]]
    fold_summary_rows.append({
        "fold": f["fold"],
        "train_start_date": f["train_start_date"],
        "train_end_date": f["train_end_date"],
        "val_start_date": f["val_start_date"],
        "val_end_date": f["val_end_date"],
        "qtd_train": len(tr),
        "qtd_val": len(va),
        "taxa_perda_train": tr[TARGET_COL].mean(),
        "taxa_perda_val": va[TARGET_COL].mean(),
    })

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "92_3g_walk_forward_folds_summary.csv")
display(fold_summary)

In [ ]:
wf_rows = []
wf_topk_rows = []
models = get_model_candidates()

print("Modelos candidatos:")
for name, prep, _ in models:
    print("-", name, "|", prep)

base_num_cols = numeric_base_cols + historical_cols

for f in folds:
    print("\n" + "="*90)
    print(f"Fold {f['fold']} | train até {f['train_end_date']} | val {f['val_start_date']} -> {f['val_end_date']}")
    tr = df_dev_sorted.loc[f["train_idx"]].copy()
    va = df_dev_sorted.loc[f["val_idx"]].copy()
    y_tr = tr[TARGET_COL].astype(int).values
    y_va = va[TARGET_COL].astype(int).values

    # Features de grafo treinadas apenas no treino do fold
    Xg_tr, Xg_va, graph_meta = make_graph_features(tr, va, graph_entity_cols, GRAPH_METHOD)
    graph_cols = list(Xg_tr.columns)

    # Encoders supervisionados treinados apenas no treino do fold
    Xce_tr, Xce_va, used_encoders = fit_transform_category_encoders(tr, y_tr, va, categorical_cols)
    ce_cols = list(Xce_tr.columns)

    # Monta cópias enriquecidas para cada fold
    tr_model = pd.concat([tr.reset_index(drop=True), Xg_tr.reset_index(drop=True), Xce_tr.reset_index(drop=True)], axis=1)
    va_model = pd.concat([va.reset_index(drop=True), Xg_va.reset_index(drop=True), Xce_va.reset_index(drop=True)], axis=1)

    hybrid_num_cols = [c for c in base_num_cols if c in tr_model.columns] + graph_cols + ce_cols
    hybrid_cat_cols = [c for c in categorical_cols if c in tr_model.columns]
    hybrid_text_cols = [c for c in text_cols if c in tr_model.columns]

    for model_name, preprocess_mode, clf in models:
        t0 = time.time()
        try:
            pre = build_preprocessor(
                preprocess_mode=preprocess_mode,
                num_cols=hybrid_num_cols,
                cat_cols=hybrid_cat_cols,
                text_cols=hybrid_text_cols,
                use_text=USE_TEXT_FEATURES,
            )
            pipe = Pipeline([
                ("pre", pre),
                ("model", clf),
            ])
            pipe.fit(tr_model, y_tr)
            proba_va = predict_proba_positive(pipe, va_model)
            metrics = eval_binary(y_va, proba_va)
            topk = topk_risco_perda_metrics(y_va, proba_va, TOP_KS)
            topk["fold"] = f["fold"]
            topk["model"] = model_name
            topk["preprocess_mode"] = preprocess_mode
            wf_topk_rows.append(topk)

            row = {
                "fold": f["fold"],
                "model": model_name,
                "preprocess_mode": preprocess_mode,
                "train_start_date": f["train_start_date"],
                "train_end_date": f["train_end_date"],
                "val_start_date": f["val_start_date"],
                "val_end_date": f["val_end_date"],
                "qtd_train": len(tr_model),
                "qtd_val": len(va_model),
                "qtd_num_features_input": len(hybrid_num_cols),
                "qtd_cat_features_input": len(hybrid_cat_cols),
                "qtd_text_cols": len(hybrid_text_cols),
                "qtd_graph_embeddings": len(graph_cols),
                "qtd_ce_features": len(ce_cols),
                "graph_embedding_method": graph_meta.get("method"),
                "graph_nodes": graph_meta.get("qtd_nodes"),
                "graph_edges_or_nnz": graph_meta.get("qtd_edges"),
                "category_encoders_used": ",".join(used_encoders),
                "fit_sec": time.time() - t0,
                **metrics,
            }
            wf_rows.append(row)
            print(f"{model_name}: PR-AUC={metrics['pr_auc_perda']:.5f} | ROC-AUC={metrics['roc_auc_perda']:.5f} | sec={row['fit_sec']:.1f}")
        except Exception as e:
            print(f"Falhou {model_name}: {repr(e)}")
        finally:
            gc.collect()

wf_results = pd.DataFrame(wf_rows)
wf_topk_results = pd.concat(wf_topk_rows, ignore_index=True) if wf_topk_rows else pd.DataFrame()

save_report(wf_results, "93_3g_hybrid_walk_forward_results.csv")
if not wf_topk_results.empty:
    save_report(wf_topk_results, "94_3g_hybrid_walk_forward_topk_results.csv")

display(wf_results.sort_values("pr_auc_perda", ascending=False).head(20))

## 9. Resumo por modelo e escolha do campeão

In [ ]:
model_summary = (
    wf_results
    .groupby(["model", "preprocess_mode"], as_index=False)
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
        mean_qtd_graph_embeddings=("qtd_graph_embeddings", "mean"),
        mean_qtd_ce_features=("qtd_ce_features", "mean"),
        mean_fit_sec=("fit_sec", "mean"),
    )
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "95_3g_hybrid_model_summary.csv")
display(model_summary)

best_row = model_summary.iloc[0]
best_model_name = best_row["model"]
best_preprocess_mode = best_row["preprocess_mode"]
print("Melhor modelo por PR-AUC médio no WF:", best_model_name)
print("Preprocessamento:", best_preprocess_mode)
print("Mean PR-AUC perda WF:", best_row["mean_pr_auc_perda"])

## 10. Treinar campeão no Dev completo e avaliar Holdout temporal

In [ ]:
# Features de grafo: fit apenas no DEV completo; aplicação no holdout usando entidades conhecidas no DEV.
Xg_dev, Xg_holdout, graph_meta_final = make_graph_features(df_dev, df_holdout, graph_entity_cols, GRAPH_METHOD)

# Encoders supervisionados: fit apenas no DEV completo; transform no holdout.
y_dev = df_dev[TARGET_COL].astype(int).values
y_holdout = df_holdout[TARGET_COL].astype(int).values
Xce_dev, Xce_holdout, used_encoders_final = fit_transform_category_encoders(df_dev, y_dev, df_holdout, categorical_cols)

# Montagem final
df_dev_model = pd.concat([df_dev.reset_index(drop=True), Xg_dev.reset_index(drop=True), Xce_dev.reset_index(drop=True)], axis=1)
df_holdout_model = pd.concat([df_holdout.reset_index(drop=True), Xg_holdout.reset_index(drop=True), Xce_holdout.reset_index(drop=True)], axis=1)

graph_cols_final = list(Xg_dev.columns)
ce_cols_final = list(Xce_dev.columns)
hybrid_num_cols_final = [c for c in base_num_cols if c in df_dev_model.columns] + graph_cols_final + ce_cols_final
hybrid_cat_cols_final = [c for c in categorical_cols if c in df_dev_model.columns]
hybrid_text_cols_final = [c for c in text_cols if c in df_dev_model.columns]

print("Qtd num/hist/graph/CE:", len(hybrid_num_cols_final))
print("Qtd categóricas:", len(hybrid_cat_cols_final))
print("Qtd text cols:", len(hybrid_text_cols_final))
print("Graph meta:", graph_meta_final)
print("Encoders usados:", used_encoders_final)

In [ ]:
# Recupera o classificador campeão
candidate_map = {name: (prep, clf) for name, prep, clf in get_model_candidates()}
preprocess_mode, clf = candidate_map[best_model_name]

pre = build_preprocessor(
    preprocess_mode=preprocess_mode,
    num_cols=hybrid_num_cols_final,
    cat_cols=hybrid_cat_cols_final,
    text_cols=hybrid_text_cols_final,
    use_text=USE_TEXT_FEATURES,
)

best_pipeline = Pipeline([
    ("pre", pre),
    ("model", clf),
])

t0 = time.time()
best_pipeline.fit(df_dev_model, y_dev)
fit_sec_final = time.time() - t0
proba_perda_holdout = predict_proba_positive(best_pipeline, df_holdout_model)

holdout_m = eval_binary(y_holdout, proba_perda_holdout)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "preprocess_mode": preprocess_mode,
    "roc_auc_perda": holdout_m["roc_auc_perda"],
    "pr_auc_perda": holdout_m["pr_auc_perda"],
    "taxa_perda_holdout": holdout_m["taxa_perda"],
    "taxa_ganho_holdout": holdout_m["taxa_ganho"],
    "best_f1_threshold_perda": holdout_m["best_f1_threshold_perda"],
    "best_f1_precision_perda": holdout_m["best_f1_precision_perda"],
    "best_f1_recall_perda": holdout_m["best_f1_recall_perda"],
    "best_f1_perda": holdout_m["best_f1_perda"],
    "precision_perda_t05": holdout_m["precision_perda_t05"],
    "recall_perda_t05": holdout_m["recall_perda_t05"],
    "f1_perda_t05": holdout_m["f1_perda_t05"],
    "pred_perda_rate_t05": holdout_m["pred_perda_rate_t05"],
    "qtd_dev": len(df_dev_model),
    "qtd_holdout": len(df_holdout_model),
    "qtd_features_input_num": len(hybrid_num_cols_final),
    "qtd_features_input_cat": len(hybrid_cat_cols_final),
    "qtd_text_cols": len(hybrid_text_cols_final),
    "qtd_graph_embeddings": len(graph_cols_final),
    "qtd_ce_features": len(ce_cols_final),
    "graph_embedding_method": graph_meta_final.get("method"),
    "graph_nodes": graph_meta_final.get("qtd_nodes"),
    "graph_edges_or_nnz": graph_meta_final.get("qtd_edges"),
    "category_encoders_used": ",".join(used_encoders_final),
    "fit_sec": fit_sec_final,
}])

save_report(holdout_metrics, "96_3g_hybrid_holdout_best_model_metrics.csv")
display(holdout_metrics)

## 11. Holdout top-k — risco puro de perda

In [ ]:
holdout_topk_perda = topk_risco_perda_metrics(
    y_true=y_holdout,
    proba_perda=proba_perda_holdout,
    ks=TOP_KS,
)
holdout_topk_perda["model"] = best_model_name
holdout_topk_perda["preprocess_mode"] = preprocess_mode
holdout_topk_perda["graph_embedding_method"] = graph_meta_final.get("method")

save_report(holdout_topk_perda, "97_3g_hybrid_holdout_topk_risco_perda.csv")
display(holdout_topk_perda)

## 12. Holdout top-k — prioridade financeira

In [ ]:
if VALUE_COL and VALUE_COL in df_holdout_model.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout_model[VALUE_COL],
        ks=TOP_KS,
    )
    holdout_topk_financeiro["model"] = best_model_name
    holdout_topk_financeiro["preprocess_mode"] = preprocess_mode
    holdout_topk_financeiro["graph_embedding_method"] = graph_meta_final.get("method")
    save_report(holdout_topk_financeiro, "98_3g_hybrid_holdout_topk_prioridade_financeira.csv")
    display(holdout_topk_financeiro)
else:
    holdout_topk_financeiro = pd.DataFrame()
    print("Nenhuma coluna de valor ajuizado encontrada. Pulando análise financeira.")

## 13. Ranking do holdout para consumo jurídico

Gera um arquivo com o score de perda, ranking de risco e ranking financeiro.

In [ ]:
ranking_cols = []
for c in [ID_COL, DATE_COL, VALUE_COL, "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "UF", "Comarca", TARGET_COL]:
    if c and c in df_holdout_model.columns and c not in ranking_cols:
        ranking_cols.append(c)

ranking_holdout = df_holdout_model[ranking_cols].copy() if ranking_cols else pd.DataFrame(index=df_holdout_model.index)
ranking_holdout[TARGET_COL] = y_holdout
ranking_holdout["proba_perda"] = proba_perda_holdout
ranking_holdout["ranking_risco_perda"] = ranking_holdout["proba_perda"].rank(method="first", ascending=False).astype(int)
ranking_holdout["percentil_risco_perda"] = ranking_holdout["proba_perda"].rank(pct=True, ascending=True)

if VALUE_COL and VALUE_COL in df_holdout_model.columns:
    ranking_holdout["valor_ajuizado_modelo"] = pd.to_numeric(df_holdout_model[VALUE_COL], errors="coerce").fillna(0).clip(lower=0)
    ranking_holdout["valor_esperado_perda"] = ranking_holdout["proba_perda"] * ranking_holdout["valor_ajuizado_modelo"]
    ranking_holdout["ranking_prioridade_financeira"] = ranking_holdout["valor_esperado_perda"].rank(method="first", ascending=False).astype(int)
else:
    ranking_holdout["valor_ajuizado_modelo"] = np.nan
    ranking_holdout["valor_esperado_perda"] = np.nan
    ranking_holdout["ranking_prioridade_financeira"] = np.nan

ranking_holdout["model"] = best_model_name
ranking_holdout["preprocess_mode"] = preprocess_mode
ranking_holdout["graph_embedding_method"] = graph_meta_final.get("method")

ranking_path = PROCESSED_DIR / "ranking_holdout_hybrid_07_03g.parquet"
ranking_holdout.to_parquet(ranking_path, index=False)
print("Ranking salvo em:", ranking_path)

save_report(ranking_holdout.sort_values("ranking_risco_perda").head(1000), "99_3g_hybrid_ranking_holdout_top1000_risco.csv")
if "ranking_prioridade_financeira" in ranking_holdout.columns:
    save_report(ranking_holdout.sort_values("ranking_prioridade_financeira").head(1000), "100_3g_hybrid_ranking_holdout_top1000_financeiro.csv")

display(ranking_holdout.sort_values("ranking_risco_perda").head(20))

## 14. Feature importance do modelo campeão

Funciona para modelos com `feature_importances_` ou `coef_`. Em pipelines com TF-IDF e OneHot, os nomes das features são extraídos do `ColumnTransformer`.

In [ ]:
def get_feature_names_from_column_transformer(ct):
    try:
        return ct.get_feature_names_out()
    except Exception:
        names = []
        for name, trans, cols in ct.transformers_:
            if name == "remainder" or trans == "drop":
                continue
            if hasattr(trans, "get_feature_names_out"):
                try:
                    sub = trans.get_feature_names_out(cols)
                except Exception:
                    sub = trans.get_feature_names_out()
                names.extend([f"{name}__{x}" for x in sub])
            else:
                names.extend([f"{name}__{c}" for c in cols])
        return np.array(names)

model_obj = best_pipeline.named_steps["model"]
feature_names = get_feature_names_from_column_transformer(best_pipeline.named_steps["pre"])

importance_df = pd.DataFrame()
if hasattr(model_obj, "feature_importances_"):
    imp = model_obj.feature_importances_
    importance_df = pd.DataFrame({"feature": feature_names[:len(imp)], "importance": imp})
elif hasattr(model_obj, "coef_"):
    coef = np.ravel(model_obj.coef_)
    importance_df = pd.DataFrame({"feature": feature_names[:len(coef)], "importance": np.abs(coef), "coef": coef})
else:
    print("Modelo não expõe feature_importances_ nem coef_.")

if not importance_df.empty:
    importance_df = importance_df.sort_values("importance", ascending=False)
    save_report(importance_df, "101_3g_hybrid_feature_importance.csv")
    display(importance_df.head(50))

## 15. Comparação com notebooks anteriores

Preencha ou ajuste os valores de referência conforme os resultados finais dos notebooks anteriores. Esta tabela ajuda a decidir se o modelo híbrido realmente agregou valor incremental.

In [ ]:
previous_steps = [
    # Ajuste estes valores se necessário com os números finais dos seus notebooks.
    {"notebook": "07_03A_baseline", "model": "random_forest_onehot_tfidf", "holdout_pr_auc_perda": 0.686900, "holdout_roc_auc_perda": 0.791299, "top5_precision_perda": 0.821372, "top10_fin_share_valor_perdas": 0.446868},
    {"notebook": "07_03B_encoders", "model": "hgb_mestimate_encoder", "holdout_pr_auc_perda": 0.674578, "holdout_roc_auc_perda": 0.786607, "top5_precision_perda": 0.814992, "top10_fin_share_valor_perdas": 0.448936},
    {"notebook": "07_03C_lgbm_xgb", "model": "xgb_ordinal", "holdout_pr_auc_perda": 0.692015, "holdout_roc_auc_perda": 0.790973, "top5_precision_perda": 0.835726, "top10_fin_share_valor_perdas": 0.450479},
    {"notebook": "07_03D_text", "model": "xgb_ordinal_tfidf", "holdout_pr_auc_perda": 0.683792, "holdout_roc_auc_perda": 0.788454, "top5_precision_perda": 0.808612, "top10_fin_share_valor_perdas": 0.445822},
    {"notebook": "07_03F_graph", "model": "lgbm_ordinal_graph", "holdout_pr_auc_perda": 0.655262, "holdout_roc_auc_perda": 0.769325, "top5_precision_perda": 0.775120, "top10_fin_share_valor_perdas": 0.439041},
]

current_top5 = float(holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"].iloc[0]) if not holdout_topk_perda.empty else np.nan
current_top10_fin = float(holdout_topk_financeiro.loc[holdout_topk_financeiro["top_k"] == 0.10, "share_valor_perdas_total"].iloc[0]) if not holdout_topk_financeiro.empty else np.nan

current_step = {
    "notebook": "07_03G_hybrid_all_features",
    "model": best_model_name,
    "holdout_pr_auc_perda": float(holdout_metrics["pr_auc_perda"].iloc[0]),
    "holdout_roc_auc_perda": float(holdout_metrics["roc_auc_perda"].iloc[0]),
    "top5_precision_perda": current_top5,
    "top10_fin_share_valor_perdas": current_top10_fin,
}

comparison = pd.DataFrame(previous_steps + [current_step])
baseline = comparison.iloc[0]
comparison["delta_pr_auc_vs_3a"] = comparison["holdout_pr_auc_perda"] - baseline["holdout_pr_auc_perda"]
comparison["delta_roc_auc_vs_3a"] = comparison["holdout_roc_auc_perda"] - baseline["holdout_roc_auc_perda"]
comparison["delta_top5_precision_vs_3a"] = comparison["top5_precision_perda"] - baseline["top5_precision_perda"]
comparison["delta_top10_fin_vs_3a"] = comparison["top10_fin_share_valor_perdas"] - baseline["top10_fin_share_valor_perdas"]

save_report(comparison, "102_3g_hybrid_comparison_previous_steps.csv")
display(comparison)

## 16. Summary final

In [ ]:
summary_step_3g = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "notebook": "07_03G_hybrid_all_features_early_stage",
    "run_mode": RUN_MODE,
    "use_text_features": USE_TEXT_FEATURES,
    "use_graph_features": USE_GRAPH_FEATURES,
    "graph_method_requested": GRAPH_METHOD,
    "graph_embedding_method_final": graph_meta_final.get("method"),
    "has_lightgbm": HAS_LIGHTGBM,
    "has_xgboost": HAS_XGBOOST,
    "has_category_encoders": HAS_CATEGORY_ENCODERS,
    "has_pecanpy": HAS_PECANPY,
    "best_model_3g_walk_forward": best_model_name,
    "best_model_3g_preprocess": preprocess_mode,
    "best_model_3g_mean_pr_auc_perda_wf": float(best_row["mean_pr_auc_perda"]),
    "best_model_3g_mean_roc_auc_perda_wf": float(best_row["mean_roc_auc_perda"]),
    "holdout_pr_auc_perda": float(holdout_metrics["pr_auc_perda"].iloc[0]),
    "holdout_roc_auc_perda": float(holdout_metrics["roc_auc_perda"].iloc[0]),
    "taxa_perda_holdout": float(holdout_metrics["taxa_perda_holdout"].iloc[0]),
    "taxa_ganho_holdout": float(holdout_metrics["taxa_ganho_holdout"].iloc[0]),
    "qtd_features_input_num": len(hybrid_num_cols_final),
    "qtd_features_input_cat": len(hybrid_cat_cols_final),
    "qtd_text_cols": len(hybrid_text_cols_final),
    "qtd_graph_embeddings": len(graph_cols_final),
    "qtd_ce_features": len(ce_cols_final),
    "qtd_dev": len(df_dev_model),
    "qtd_holdout": len(df_holdout_model),
    "ranking_risco_holdout_path": str(PROCESSED_DIR / "ranking_holdout_hybrid_07_03g.parquet"),
}])

save_report(summary_step_3g, "103_summary_step_3g_hybrid_all_features.csv")
display(summary_step_3g.T)

## O que verificar após rodar

1. Se o **07_03G** superou o **07_03C** em PR-AUC no holdout.
2. Se houve ganho em **Precision@Top 5%**.
3. Se houve ganho em **share de valor perdido capturado no Top 10% financeiro**.
4. Se o grafo entrou entre as features importantes ou se foi pouco usado pelo modelo.
5. Se o ganho compensa o custo operacional de manter Node2Vec/PecanPy no pipeline.

Minha recomendação de decisão:

- Se o 07_03G melhorar PR-AUC e top-k financeiro, ele vira novo candidato campeão.
- Se melhorar apenas top-k financeiro, ele pode ser usado para ranking financeiro, mantendo outro modelo para risco puro.
- Se não melhorar, manter o grafo como produto complementar de similaridade/explicabilidade, não como feature produtiva do modelo principal.